# 11 · Cascade: ASR → VQA (T1_Speech)

ASR transcripts replace the text question as input to the T2_Text VQA model.
Same model, prompt and ids as the T2_Text run, so the only difference is the question source.
Runs on a Kaggle GPU with `MMCQA_OUT=/kaggle/working` and the transcripts copied to `/kaggle/working/asr/`.

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared").is_dir())
sys.path[:0] = [str(ROOT / "shared"), str(ROOT / "t1_speech")]

from data import *
from metrics import *
from speech import *

In [ ]:
from vqa import answer

SPLIT = "dev"
ASR_MODEL = "whisper-large-v3-turbo"
ASR_SHORT = "whisperturbo"
VQA_MODEL = "qwen25vl7b"
PROMPT_ID = "p2"        
BATCH = 8

In [ ]:
rows = cascade_inputs(asr_path(SPLIT, ASR_MODEL), SPLIT)
out = preds_path("t1_speech_cascade", SPLIT, f"{ASR_SHORT}+{VQA_MODEL}", PROMPT_ID)
print(len(rows), "questions ->", out)
print("text:", rows[0]["question_text"])
print("asr: ", rows[0]["question"])

## Run (resumable, appends after every batch)

In [ ]:
from tqdm.auto import tqdm

done = {r["id"] for r in load_jsonl(out)} if out.exists() else set()
todo = [r for r in rows if r["id"] not in done]
print(f"{len(done)} done, {len(todo)} to go")

for i in tqdm(range(0, len(todo), BATCH)):
    b = todo[i:i + BATCH]
    preds = answer([r["image_path"] for r in b], [r["question"] for r in b], prompt_id=PROMPT_ID)
    save_jsonl([{"id": r["id"], "answer": p, "lang": LANG} for r, p in zip(b, preds)], out, append=True)

## Score

In [ ]:
res = evaluate(out, SPLIT)
res

Check that the T2_Text run covers the same ids.

In [ ]:
text_out = preds_path("t2_text", SPLIT, VQA_MODEL, PROMPT_ID)
if text_out.exists():
    a = {r["id"] for r in load_jsonl(out)}
    b = {r["id"] for r in load_jsonl(text_out)}
    print(f"speech {len(a)}, text {len(b)}, common {len(a & b)}")
else:
    print("no T2_Text preds yet:", text_out)